# Lesson 5: The secret of AI writing — predicting the next word 🔮
**Learning goals**
1. Understand the core idea of language models: **guess the next word from the words before it**
2. Build your own "AI nursery rhyme writer" from scratch
3. See how a real language model scores every possible next word

**⏱️ Timing:** counting 15' | rhyme writer 20' | temperature 10' | real model 15'

## 📱 An analogy first
When you text on your phone and type "see you", the keyboard suggests "tomorrow", "soon", "later".
How does it know? It has seen lots of people type, and **counted** what usually comes after "see you".

ChatGPT is basically a **super-powered autocomplete**: guess one word, add it, guess the next one…

In [ ]:
corpus = """Twinkle, twinkle, little star, how I wonder what you are. Up above the world so high, like a diamond in the sky.
Mary had a little lamb, its fleece was white as snow. And everywhere that Mary went, the lamb was sure to go.
Humpty Dumpty sat on a wall. Humpty Dumpty had a great fall. All the king's horses and all the king's men could not put Humpty together again.
Jack and Jill went up the hill to fetch a pail of water. Jack fell down and broke his crown, and Jill came tumbling after.
Hey diddle diddle, the cat and the fiddle, the cow jumped over the moon. The little dog laughed to see such sport, and the dish ran away with the spoon.
Row, row, row your boat, gently down the stream. Merrily, merrily, merrily, merrily, life is but a dream.
Baa, baa, black sheep, have you any wool? Yes sir, yes sir, three bags full. One for the master, one for the dame, and one for the little boy who lives down the lane.
Little Bo Peep has lost her sheep and does not know where to find them. Leave them alone and they will come home, wagging their tails behind them.
Hickory dickory dock, the mouse ran up the clock. The clock struck one, the mouse ran down. Hickory dickory dock.
Rain, rain, go away, come again another day. Little Johnny wants to play.
Old Mother Hubbard went to the cupboard to fetch her poor dog a bone. When she got there the cupboard was bare, and so the poor dog had none.
Little Miss Muffet sat on a tuffet, eating her curds and whey. Along came a spider who sat down beside her and frightened Miss Muffet away.
Little Jack Horner sat in the corner, eating his Christmas pie. He put in his thumb and pulled out a plum and said, what a good boy am I.
The itsy bitsy spider went up the water spout. Down came the rain and washed the spider out. Out came the sun and dried up all the rain, and the itsy bitsy spider went up the spout again."""
import re
words = re.findall(r"[a-z']+|[.,?]", corpus.lower())

def join(ws):
    """Glue words back into text (no space before punctuation)."""
    return " ".join(ws).replace(" .", ".").replace(" ,", ",").replace(" ?", "?")

print("Our training data has", len(words), "words")
print(words[:20])

## Step 1: Count "who comes after whom"
For example, after "little" we've seen "star", "lamb", "dog"… Let's count all of them.

In [ ]:
from collections import defaultdict, Counter

next_word = defaultdict(Counter)
for a, b in zip(words, words[1:]):
    next_word[a][b] += 1

for w in ["little", "the", "humpty", "went"]:
    print(f"After '{w}' we often see:", next_word[w].most_common(5))

### 🧩 Exercise 1
1. Find out what usually comes after "mary" and "down".
2. Think: why does "the" have so many different words after it?

In [ ]:
# your code

## Step 2: The AI rhyme writer ✍️
Start from one word, then pick the next word **by probability** (words seen more often are more likely to be chosen), and keep going.

In [ ]:
import random

def write_rhyme(start="little", length=25):
    text = [start]
    for _ in range(length - 1):
        options = next_word[text[-1]]
        if not options:            # never seen anything after this word → stop
            break
        ws, counts = zip(*options.items())
        text.append(random.choices(ws, weights=counts)[0])
    return join(text)

for s in ["little", "the", "jack", "rain"]:
    print(write_rhyme(s), "\n")

Read them out loud! Some bits sound like a real rhyme, some are nonsense. **Why?**
👉 It only looks at the **one previous word** — its memory is one word long. Real large models can look back over **thousands or even hundreds of thousands of words**.

### 🧩 Exercise 2
1. Run it a few more times and pick the best line. Write it down.
2. Replace `corpus` with your own text (an essay you wrote, a page from a book you own, a school notice) and run Steps 1 and 2 again.

## Step 3: "Temperature" — should the AI be bold or careful? 🌡️
- **Low temperature:** always picks the most common word → safe but boring, often repeats
- **High temperature:** rare words get a chance too → creative but more likely to talk nonsense

(This is why the same question gave different answers in Lesson 2!)

In [ ]:
def write_with_temp(start="the", length=25, temperature=1.0):
    text = [start]
    for _ in range(length - 1):
        options = next_word[text[-1]]
        if not options: break
        ws, counts = zip(*options.items())
        weights = [c ** (1 / temperature) for c in counts]   # low temperature → big counts get even bigger
        text.append(random.choices(ws, weights=weights)[0])
    return join(text)

for t in [0.1, 1.0, 5.0]:
    print(f"temperature {t}:", write_with_temp("the", temperature=t), "\n")

### 🚀 Challenge: a longer memory
Change "previous 1 word" to "previous 2 words" (a trigram model). Hint: `next2[(words[i], words[i+1])][words[i+2]] += 1`. Is the text more sensible? Why does it start **copying the original text** when the data is small?

In [ ]:
# challenge area

## Step 4: Meet a real language model 🤖
Real models don't work word by word but **token** by token. Let's see how one splits text into tokens:

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tok = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

for s in ["I love pizza", "unbelievable", "Supercalifragilistic"]:
    print(s, "→", tok.tokenize(s))

In [ ]:
def top_next_words(text, k=10):
    ids = tok(text, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(ids).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    for p, i in zip(top.values, top.indices):
        print(f"{p:6.1%}  {tok.decode(i)!r}")

top_next_words("The cat sat on the")

### 🧩 Exercise 3
Try these beginnings and see what the AI thinks comes next:
- `"My favourite food is"`
- `"The capital of France is"`
- `"Once upon a"`

**Think:** when the AI says the capital of France, does it really "know", or has it just "seen it a lot"? (We'll come back to this in Lesson 14.)

In [ ]:
top_next_words("Once upon a")

## 📌 Summary
- A language model = **predict the next token**, one after another
- Our rhyme writer remembers only 1 word; big models remember very long contexts
- **Temperature** controls whether the AI plays it safe or gets creative
- **Next lesson:** neural networks with a longer memory — RNNs, LSTMs and GRUs